# Historical Knowledge Graph QA Pipeline (Qwen Only)

This notebook implements a Question Answering pipeline using a Knowledge Graph (Neo4j) with **Qwen model only**:
- **Qwen (Local)**: Uses `Qwen/Qwen3-4B` for local inference
- **Multi-GPU Support**: Uses `device_map="auto"` for automatic GPU distribution

## Key Features
- Improved answer normalization for MCQ and True/False questions
- Robust entity extraction for Vietnamese historical content
- Hybrid retrieval from Knowledge Graph

## Setup
1. Add `NEO4J_PASSWORD` to Kaggle Secrets
2. Ensure the input files are at `/kaggle/input/historical/`

In [1]:
# Install dependencies
!pip install -q transformers torch accelerate neo4j bitsandbytes sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 92.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 74.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 32.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 14.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 76.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.4/325.4 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━

In [ ]:
import json
import logging
import os
import re
import time
import torch
import numpy as np
from typing import List, Dict, Any, Tuple, Optional
from tqdm import tqdm
from datetime import datetime

# Kaggle secrets
try:
    from kaggle_secrets import UserSecretsClient
    _KAGGLE_SECRETS_AVAILABLE = True
except Exception:
    _KAGGLE_SECRETS_AVAILABLE = False

def get_secret(key: str, default: str = None) -> str:
    if _KAGGLE_SECRETS_AVAILABLE:
        try:
            user_secrets = UserSecretsClient()
            val = user_secrets.get_secret(key)
            if val is not None and val != "":
                return val
        except Exception:
            pass
    val = os.getenv(key)
    if val:
        return val
    return default

# Logging setup
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger("GraphRAG")

# GPU Info
print("GPU Information:")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"         Memory: {torch.cuda.get_device_properties(i).total_memory / 1024**3:.1f} GB")
else:
    print("  No GPU available")

print("\nLibraries loaded successfully!")

In [ ]:
# ================================================================================
# CONFIGURATION
# ================================================================================

# Model Configuration
QWEN_MODEL = "Qwen/Qwen3-4B"
EMBEDDING_MODEL = "Qwen/Qwen3-Embedding-0.6B"
RERANKER_MODEL = "Qwen/Qwen3-Reranker-0.6B"

# Multi-GPU Support
DEVICE_MAP = "auto"  # "auto" for multi-GPU, "cuda:0" for single GPU

# File paths
QUESTION_FILE = "/kaggle/input/sample_20question.json"
KG_FILE = "/kaggle/input/knowledge_graph_historical_v5.json"
ENTITIES_FILE = "/kaggle/input/historical/entities_v5.json"
OUTPUT_FILE = "/kaggle/working/results_qwen_only.json"
REPORT_FILE = "/kaggle/working/evaluation_report.html"

# Neo4j Configuration
NEO4J_URI = os.getenv("NEO4J_URI", "neo4j+s://5f398723.databases.neo4j.io")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = get_secret("NEO4J_PASSWORD", "password")

print("Configuration loaded!")
print(f"  - Qwen Model: {QWEN_MODEL}")
print(f"  - Device Map: {DEVICE_MAP}")
print(f"  - Neo4j URI: {NEO4J_URI}")

In [ ]:
# ================================================================================
# Import GraphRAG package
# ================================================================================

import sys
sys.path.insert(0, '/kaggle/input')

from graphrag import (
    GraphRAGConfig,
    EmbeddingGenerator,
    Reranker,
    Neo4jManager,
    HybridRetriever,
    ContextBuilder,
    EntityExtractor,
    AnswerGenerator,
    logger
)

print("GraphRAG package imported successfully!")

In [ ]:
# ================================================================================
# Initialize Components
# ================================================================================

# Create configuration
config = GraphRAGConfig(
    neo4j_uri=NEO4J_URI,
    neo4j_user=NEO4J_USER,
    neo4j_password=NEO4J_PASSWORD,
    qwen_model=QWEN_MODEL,
    embedding_model=EMBEDDING_MODEL,
    reranker_model=RERANKER_MODEL,
    questions_file=QUESTION_FILE,
    kg_file=KG_FILE,
    output_file=OUTPUT_FILE,
)

print("Configuration created!")

In [ ]:
# ================================================================================
# Initialize Entity Extractor
# ================================================================================

entity_extractor = EntityExtractor()
print("Entity Extractor initialized!")

In [ ]:
# ================================================================================
# Initialize Embedding Generator and Reranker
# ================================================================================

embedding_gen = EmbeddingGenerator(config)
print("Embedding Generator initialized!")

reranker = Reranker(config)
print("Reranker initialized!")

In [ ]:
# ================================================================================
# Initialize Neo4j Connection
# ================================================================================

neo4j_connected = False
neo4j_manager = None

try:
    neo4j_manager = Neo4jManager(config)
    neo4j_connected = True
    print("Neo4j connected successfully!")
except Exception as e:
    print(f"Neo4j connection failed: {e}")
    print("Continuing without Neo4j (LLM-only mode)")

In [ ]:
# ================================================================================
# Initialize Retriever and Context Builder
# ================================================================================

if neo4j_connected:
    retriever = HybridRetriever(config, embedding_gen, neo4j_manager, reranker)
    context_builder = ContextBuilder(config)
    print("Retriever and Context Builder initialized!")
else:
    retriever = None
    context_builder = None
    print("Retriever not available (no Neo4j connection)")

In [ ]:
# ================================================================================
# Initialize Answer Generator (Qwen Only with Multi-GPU)
# ================================================================================

print("\n" + "="*60)
print("INITIALIZING QWEN ANSWER GENERATOR")
print(f"Device Map: {DEVICE_MAP}")
print("="*60)

answer_gen = AnswerGenerator(
    config=config,
    model_name=QWEN_MODEL,
    device_map=DEVICE_MAP
)

print("\nQwen Answer Generator initialized!")
print(f"  - Model: {QWEN_MODEL}")
print(f"  - Device Map: {DEVICE_MAP}")

print("\nAll components initialized!")

In [ ]:
# ================================================================================
# Question Processing Function
# ================================================================================

def process_question(q_data: Dict) -> Dict:
    """Process a single question using Qwen model."""
    start_time = time.time()
    
    question = q_data.get("question", "")
    options = q_data.get("options", [])
    
    # Determine question type
    is_tf = False
    if len(options) == 2:
        answers = [opt.get("answer", "").lower() for opt in options]
        if "đúng" in answers and "sai" in answers:
            is_tf = True
    if not is_tf and re.search(r'đúng hay sai|phát biểu.*đúng', question.lower()):
        is_tf = True
    
    question_type = "tf" if is_tf else "mcq"
    
    # Get correct answer
    correct_answer = ""
    for i, opt in enumerate(options):
        if opt.get("isCorrect", False):
            if question_type == "tf":
                correct_answer = opt.get("answer", "")
            else:
                correct_answer = chr(ord('A') + i)
            break
    
    # Extract entities
    entities = []
    if is_tf:
        statement = entity_extractor.extract_tf_statement(question)
        entities = entity_extractor.extract(statement)
    else:
        option_texts = [opt.get("answer", "") for opt in options]
        entities = entity_extractor.extract(question, option_texts)
    
    # Get context from Neo4j (if available)
    context = ""
    provenance = []
    candidates = []
    if retriever and context_builder:
        candidates = retriever.retrieve(entities, question, top_k=config.top_k_retrieval)
        context, provenance = context_builder.build_context(candidates, token_budget=5000)
    
    # Generate answer using Qwen
    if is_tf:
        statement = entity_extractor.extract_tf_statement(question)
        source_materials = entity_extractor.extract_source_materials(question)
        model_answer, raw_response = answer_gen.answer_tf(
            statement, context, source_materials=source_materials
        )
    else:
        option_texts = [opt.get("answer", "") for opt in options]
        model_answer, raw_response = answer_gen.answer_mcq(question, option_texts, context)
    
    # Compare answers
    def compare(model_ans, correct_ans, q_type):
        if q_type == "tf":
            return ("đúng" in model_ans.lower()) == ("đúng" in correct_ans.lower())
        return model_ans.strip().upper() == correct_ans.strip().upper()
    
    is_correct = compare(model_answer, correct_answer, question_type)
    
    processing_time = time.time() - start_time
    
    # Calculate trust score
    scores = [c.get("combined_score", 0) for c in candidates[:3]]
    if scores and isinstance(scores[0], (list, tuple)):
        scores = [s[0] if s else 0 for s in scores]
    avg_score = sum(scores) / len(scores) if scores else 0
    
    return {
        "question": question,
        "question_type": question_type,
        "correct_answer": correct_answer,
        "model_answer": model_answer,
        "is_correct": is_correct,
        "raw_response": raw_response,
        "entities_extracted": entities,
        "context_length": len(context),
        "num_candidates": len(candidates),
        "trust_score": round(avg_score, 4),
        "trust_level": "high" if avg_score > 0.7 else "medium" if avg_score > 0.5 else "low",
        "evidence": [{
            "source": p.get("source"),
            "score": round(p.get("score", 0), 4),
            "provenance": p.get("provenance"),
            "text_preview": (p.get("text_preview", "") or "")[:200]
        } for p in provenance[:10]],
        "processing_time": round(processing_time, 2)
    }

print("Question processing function ready!")

In [ ]:
# ================================================================================
# Load Questions
# ================================================================================

with open(QUESTION_FILE, 'r', encoding='utf-8') as f:
    data = json.load(f)

questions = []
if isinstance(data, list):
    questions = data
else:
    questions.extend(data.get("multiple_choice", []))
    questions.extend(data.get("true_false", []))

print(f"Loaded {len(questions)} questions")

# Optional: limit for testing
# questions = questions[:20]

In [ ]:
# ================================================================================
# Process All Questions
# ================================================================================

results = []
stats = {
    "total": 0,
    "correct": 0,
    "mcq": 0, "mcq_correct": 0,
    "tf": 0, "tf_correct": 0
}

start_time = time.time()

for i, q in enumerate(tqdm(questions, desc="Processing")):
    try:
        result = process_question(q)
        results.append(result)
        
        # Update stats
        stats["total"] += 1
        q_type = result["question_type"]
        stats[q_type] += 1
        
        if result["is_correct"]:
            stats["correct"] += 1
            stats[f"{q_type}_correct"] += 1
        
        # Progress update every 10 questions
        if (i + 1) % 10 == 0 or i == len(questions) - 1:
            total_acc = stats["correct"] / stats["total"] * 100
            mcq_acc = stats["mcq_correct"] / stats["mcq"] * 100 if stats["mcq"] > 0 else 0
            tf_acc = stats["tf_correct"] / stats["tf"] * 100 if stats["tf"] > 0 else 0
            print(f"\n[{i+1}/{len(questions)}] Total: {total_acc:.1f}% | MCQ: {mcq_acc:.1f}% | T/F: {tf_acc:.1f}%")
            
    except Exception as e:
        print(f"Error on Q{i+1}: {e}")
        import traceback
        traceback.print_exc()

total_time = time.time() - start_time
print(f"\nTotal processing time: {total_time:.1f}s")

In [ ]:
# ================================================================================
# Final Statistics
# ================================================================================

print("\n" + "="*70)
print("FINAL RESULTS - QWEN ONLY")
print("="*70)

# Overall stats
total = stats["total"]
total_acc = stats["correct"] / total * 100 if total > 0 else 0

print(f"\nOVERALL ACCURACY:")
print(f"  TOTAL: {stats['correct']}/{total} = {total_acc:.2f}%")

# MCQ stats
if stats["mcq"] > 0:
    mcq_acc = stats["mcq_correct"] / stats["mcq"] * 100
    print(f"\nMCQ ACCURACY:")
    print(f"  {stats['mcq_correct']}/{stats['mcq']} = {mcq_acc:.2f}%")

# T/F stats
if stats["tf"] > 0:
    tf_acc = stats["tf_correct"] / stats["tf"] * 100
    print(f"\nTRUE/FALSE ACCURACY:")
    print(f"  {stats['tf_correct']}/{stats['tf']} = {tf_acc:.2f}%")

# Trust level distribution
high_trust = sum(1 for r in results if r.get("trust_level") == "high")
medium_trust = sum(1 for r in results if r.get("trust_level") == "medium")
low_trust = sum(1 for r in results if r.get("trust_level") == "low")

print(f"\nTRUST LEVEL DISTRIBUTION:")
print(f"  High:   {high_trust} ({high_trust/total*100:.1f}%)")
print(f"  Medium: {medium_trust} ({medium_trust/total*100:.1f}%)")
print(f"  Low:    {low_trust} ({low_trust/total*100:.1f}%)")

# Timing
avg_time = total_time / total if total > 0 else 0
print(f"\nTIMING:")
print(f"  Total: {total_time:.1f}s")
print(f"  Avg per question: {avg_time:.2f}s")

print("\n" + "="*70)

In [ ]:
# ================================================================================
# Save Results
# ================================================================================

output = {
    "generated_at": datetime.now().isoformat(),
    "pipeline": "qwen-only",
    "model": QWEN_MODEL,
    "stats": {
        "total": stats["total"],
        "correct": stats["correct"],
        "accuracy": round(total_acc, 2),
        "mcq": stats["mcq"],
        "mcq_correct": stats["mcq_correct"],
        "mcq_accuracy": round(stats["mcq_correct"] / stats["mcq"] * 100, 2) if stats["mcq"] > 0 else 0,
        "tf": stats["tf"],
        "tf_correct": stats["tf_correct"],
        "tf_accuracy": round(stats["tf_correct"] / stats["tf"] * 100, 2) if stats["tf"] > 0 else 0,
    },
    "timing": {
        "total_seconds": round(total_time, 2),
        "avg_per_question": round(avg_time, 2)
    },
    "config": {
        "qwen_model": QWEN_MODEL,
        "embedding_model": EMBEDDING_MODEL,
        "reranker_model": RERANKER_MODEL,
        "device_map": DEVICE_MAP,
        "neo4j_connected": neo4j_connected
    },
    "results": results
}

with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"Results saved to: {OUTPUT_FILE}")

In [ ]:
# ================================================================================
# Generate HTML Report
# ================================================================================

html = f"""<!DOCTYPE html>
<html lang="vi">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>GraphRAG QA Report - Qwen Only</title>
    <style>
        body {{ font-family: 'Segoe UI', Arial, sans-serif; margin: 20px; background: #f5f5f5; }}
        .header {{ background: linear-gradient(135deg, #667eea, #764ba2); color: white; padding: 25px; border-radius: 12px; margin-bottom: 25px; }}
        .header h1 {{ margin: 0 0 10px 0; }}
        .stats-grid {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(180px, 1fr)); gap: 15px; margin-bottom: 30px; }}
        .stat-card {{ background: white; padding: 20px; border-radius: 10px; text-align: center; box-shadow: 0 2px 8px rgba(0,0,0,0.1); }}
        .stat-card.primary {{ border-top: 4px solid #3498db; }}
        .stat-card.success {{ border-top: 4px solid #27ae60; }}
        .stat-card.warning {{ border-top: 4px solid #f39c12; }}
        .stat-value {{ font-size: 32px; font-weight: bold; color: #2c3e50; }}
        .stat-label {{ color: #7f8c8d; margin-top: 5px; }}
        .question-card {{ background: white; padding: 20px; margin-bottom: 15px; border-radius: 10px; box-shadow: 0 2px 8px rgba(0,0,0,0.08); }}
        .question-header {{ display: flex; justify-content: space-between; margin-bottom: 15px; align-items: center; }}
        .badges {{ display: flex; gap: 10px; }}
        .badge {{ padding: 5px 12px; border-radius: 15px; font-size: 12px; font-weight: bold; }}
        .badge.correct {{ background: #d5f4e6; color: #27ae60; }}
        .badge.wrong {{ background: #fadbd8; color: #e74c3c; }}
        .badge.mcq {{ background: #d6eaf8; color: #2980b9; }}
        .badge.tf {{ background: #e8daef; color: #8e44ad; }}
        .badge.high {{ background: #d5f4e6; color: #27ae60; }}
        .badge.medium {{ background: #fef9e7; color: #f39c12; }}
        .badge.low {{ background: #fadbd8; color: #e74c3c; }}
        .question-text {{ margin-bottom: 15px; line-height: 1.7; }}
        .answer-row {{ display: flex; gap: 20px; margin-top: 10px; padding: 10px; background: #f8f9fa; border-radius: 8px; }}
        .answer-item {{ flex: 1; }}
        .answer-label {{ font-weight: bold; color: #7f8c8d; font-size: 12px; }}
    </style>
</head>
<body>
    <div class="header">
        <h1>GraphRAG QA Evaluation Report</h1>
        <p>Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
        <p>Model: {QWEN_MODEL} | Device: {DEVICE_MAP}</p>
    </div>
    
    <div class="stats-grid">
        <div class="stat-card primary">
            <div class="stat-value">{total_acc:.1f}%</div>
            <div class="stat-label">Overall Accuracy</div>
        </div>
        <div class="stat-card success">
            <div class="stat-value">{stats['correct']}/{total}</div>
            <div class="stat-label">Correct Answers</div>
        </div>
        <div class="stat-card">
            <div class="stat-value">{stats['mcq_correct']}/{stats['mcq']}</div>
            <div class="stat-label">MCQ Correct</div>
        </div>
        <div class="stat-card">
            <div class="stat-value">{stats['tf_correct']}/{stats['tf']}</div>
            <div class="stat-label">T/F Correct</div>
        </div>
        <div class="stat-card warning">
            <div class="stat-value">{avg_time:.2f}s</div>
            <div class="stat-label">Avg per Question</div>
        </div>
    </div>
"""

# Add question details
html += "<h2>Question Details</h2>\n"

for i, r in enumerate(results[:100], 1):  # Limit to 100 for HTML size
    q_type = r['question_type'].upper()
    status = 'correct' if r['is_correct'] else 'wrong'
    trust = r.get('trust_level', 'low')
    
    html += f"""
    <div class="question-card">
        <div class="question-header">
            <strong>Q{i}</strong>
            <div class="badges">
                <span class="badge {q_type.lower()}">{q_type}</span>
                <span class="badge {status}">{r['model_answer']}</span>
                <span class="badge">Correct: {r['correct_answer']}</span>
                <span class="badge {trust}">Trust: {trust}</span>
            </div>
        </div>
        <div class="question-text">{r['question'][:400]}{'...' if len(r['question']) > 400 else ''}</div>
        <div class="answer-row">
            <div class="answer-item">
                <div class="answer-label">Entities Extracted</div>
                <div>{', '.join(r.get('entities_extracted', [])[:5]) or 'None'}</div>
            </div>
            <div class="answer-item">
                <div class="answer-label">Context Length</div>
                <div>{r.get('context_length', 0)} chars</div>
            </div>
            <div class="answer-item">
                <div class="answer-label">Processing Time</div>
                <div>{r.get('processing_time', 0)}s</div>
            </div>
        </div>
    </div>
"""

html += """</body></html>"""

with open(REPORT_FILE, 'w', encoding='utf-8') as f:
    f.write(html)

print(f"HTML report saved to: {REPORT_FILE}")

In [ ]:
# ================================================================================
# Analyze Incorrect Answers (Optional)
# ================================================================================

# Get incorrect answers
incorrect = [r for r in results if not r['is_correct']]

print(f"\n{'='*70}")
print(f"INCORRECT ANSWER ANALYSIS ({len(incorrect)} questions)")
print(f"{'='*70}")

# By question type
incorrect_mcq = [r for r in incorrect if r['question_type'] == 'mcq']
incorrect_tf = [r for r in incorrect if r['question_type'] == 'tf']

print(f"\nIncorrect MCQ: {len(incorrect_mcq)}")
print(f"Incorrect T/F: {len(incorrect_tf)}")

# By trust level
incorrect_high = [r for r in incorrect if r.get('trust_level') == 'high']
incorrect_medium = [r for r in incorrect if r.get('trust_level') == 'medium']
incorrect_low = [r for r in incorrect if r.get('trust_level') == 'low']

print(f"\nIncorrect by Trust Level:")
print(f"  High trust:   {len(incorrect_high)} (confidence was wrong)")
print(f"  Medium trust: {len(incorrect_medium)}")
print(f"  Low trust:    {len(incorrect_low)} (expected - low confidence)")

# Sample some incorrect answers
print(f"\n{'='*70}")
print("SAMPLE INCORRECT ANSWERS (first 3):")
print(f"{'='*70}")

for i, r in enumerate(incorrect[:3], 1):
    print(f"\n--- Incorrect #{i} ({r['question_type'].upper()}) ---")
    print(f"Question: {r['question'][:200]}...")
    print(f"Model Answer: {r['model_answer']}")
    print(f"Correct Answer: {r['correct_answer']}")
    print(f"Trust: {r.get('trust_level', 'N/A')}")
    print(f"Entities: {r.get('entities_extracted', [])[:5]}")

In [ ]:
import json

def filter_incorrect_items(path_in, path_out):
    with open(path_in, "r", encoding="utf-8") as f:
        data = json.load(f)

    results = data.get("results", [])

    filtered = []
    for item in results:
        # lọc những mục mà model trả lời sai -> item["is_correct"] == False
        if not item.get("is_correct", True):
            filtered.append({
                "question": item.get("question"),
                "question_type": item.get("question_type"),
                "correct_answer": item.get("correct_answer"),
                "model_answer": item.get("model_answer"),
                "raw_response": item.get("raw_response"),
                "is_correct": item.get("is_correct"),
                "context_length": item.get("context_length"),
                "trust_level": item.get("trust_level"),
            })

    with open(path_out, "w", encoding="utf-8") as f:
        json.dump({"results": filtered}, f, ensure_ascii=False, indent=2)

    return filtered

# ví dụ:
filtered = filter_incorrect_items("/kaggle/working/results_qwen_only.json", "incorrect_filtered.json")


In [ ]:
# ================================================================================
# Cleanup
# ================================================================================

if neo4j_manager:
    neo4j_manager.close()
    print("Neo4j connection closed.")

print("\nPipeline completed successfully!")
print(f"\nOutput files:")
print(f"  - Results JSON: {OUTPUT_FILE}")
print(f"  - HTML Report:  {REPORT_FILE}")